# Piloto 2026 — recuperación aromática dependiente de etanol

## tl;dr

**Veredicto:** `NO_VALID_MODEL`.  
**Modelo seleccionado:** `None`.

Se compara una fracción efectiva constante con `eta(E)=eta50·m^((E−50)/10)`. El cambio de etanol modifica sólo el operador de recuperación del condensado; la producción, la partición gas/líquido y el balance de masa permanecen iguales.

## Alcance y supuestos

- El etanol del vino es un proxy de la composición del vapor/condensado; no se midió etanol en cada MIX.
- `eta(E)` agrega un parámetro respecto del modelo de captura constante y se evalúa contra éste, no contra el supuesto nominal 0,85–0,88.
- La muestra 26211-P-12 se conserva en el análisis primario.
- El análisis es post hoc e internamente validado por reactor; requiere confirmación prospectiva y calibración de trampa.

In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import display, Image

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'fermentation_model').exists():
    ROOT = ROOT.parent
if not (ROOT / 'fermentation_model').exists():
    raise RuntimeError('Execute from the repository or a descendant directory')
sys.path.insert(0, str(ROOT / 'fermentation_model'))
from pilot_2026 import run_aroma_ethanol_capture_validation_2026 as analysis
result = analysis.load_results() if os.environ.get('PILOT_AROMA_REUSE_RESULTS') == '1' else analysis.run_analysis()
print('Veredicto:', result['gate']['verdict'])
print('Comparador:', result['gate']['comparison_baseline_model'])
print('Seleccionado:', result['gate']['selected_model'])

## Validación cruzada

In [ ]:
display(result['metrics'].round(4))
display(result['comparison'].round(4))
display(Image(filename=analysis.FIGURE_DIR / '02_capture_efficiency_loro_nrmse.png'))
display(Image(filename=analysis.FIGURE_DIR / '07_capture_efficiency_by_fold.png'))

## CO₂, temperatura, pulso y curvas

In [ ]:
display(Image(filename=analysis.FIGURE_DIR / '03_rco2_temperature_pulse_drivers.png'))
for species in analysis.capture.base.SPECIES_LABELS:
    display(Image(filename=analysis.FIGURE_DIR / f'04_liquid_{species}.png'))
    display(Image(filename=analysis.FIGURE_DIR / f'05_condensate_{species}.png'))

## Estabilidad

In [ ]:
display(result['stability'].round(5))
display(result['parameters'].round(6))
display(result['fit_validation'].round(5))

## Takeaways

- Una mejora transferible apoyaría que el cambio de composición del condensado es parte del error temporal.
- Un multiplicador en el límite o inestable entre folds implica que etanol sólo está actuando como proxy de tiempo.
- Incluso con `PASS`, la validación confirmatoria debe medir etanol/volumen de cada MIX, concentración en gas de salida y recuperación de un estándar gaseoso.

In [ ]:
assert result['gate']['verdict'] in {'PASS', 'NO_VALID_MODEL'}
assert len(result['figures']) == 8
assert result['fit_validation']['maximum_relative_mass_balance_error'].max() <= 1e-8
print('Notebook ejecutado sin errores; figuras:', len(result['figures']))